# Fase 4: tiempos de memoria constante frente a global

Mide `build/hough` (global) y `build/hough_const` (global + constante) en la **misma sesión y GPU**, con el protocolo de la fase 2: un calentamiento excluido y diez mediciones del kernel con CUDA Events, validando cada ejecución contra la CPU.

Una primera prueba mostró que los tiempos bajan durante la sesión mientras la GPU sale de reposo (P8), y que ese efecto supera la diferencia entre versiones (`results/fase4-constante/exploratorio/`). Por eso aquí se estabiliza la GPU con 2000 ejecuciones descartadas y se alternan cuatro rondas en orden ABBA (global-constante, constante-global, ...), registrando el estado de la GPU antes de cada ejecución. Usa **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

In [ ]:
!nvidia-smi
!nvcc --version

## Proyecto

In [ ]:
from pathlib import Path
import subprocess, os
RAMA = 'main'
root = Path('/content/P3-CUDA-Paralela')
if not root.exists():
    subprocess.run(['git', 'clone', '--branch', RAMA,
                    'https://github.com/Jorge162017/P3-CUDA-Paralela.git', str(root)], check=True)
os.chdir(root)
subprocess.run(['git', 'fetch', 'origin', RAMA], check=True)
subprocess.run(['git', 'checkout', RAMA], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', RAMA], check=True)
subprocess.run(['make', 'build/hough', 'build/hough_const'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Rama:', RAMA, '| Commit:', commit)

## Diez mediciones por versión y ronda

Cada ejecución escribe su CSV en `build/`; solo se copian los `*-times.csv` a `results/fase4-constante/estable/`. El log se guarda aunque falle una ejecución.

In [ ]:
import shutil
import pandas as pd
out = Path('results/fase4-constante/estable')
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
ORDEN = {'ronda1': ['global', 'constante'], 'ronda2': ['constante', 'global'],
         'ronda3': ['global', 'constante'], 'ronda4': ['constante', 'global']}
PROGRAMA = {'global': './build/hough', 'constante': './build/hough_const'}
CONSULTA = ['nvidia-smi', '--query-gpu=pstate,clocks.sm,clocks.mem,temperature.gpu,power.draw',
            '--format=csv,noheader,nounits']
log = subprocess.check_output(['nvidia-smi'], text=True)
log += subprocess.check_output(['nvcc', '--version'], text=True)
log += f'Rama: {RAMA}\nCommit: {commit}\n'
log += 'Compilacion: make build/hough build/hough_const; -O2 -std=c++17 --fmad=false\n'
log += 'Protocolo: 2000 ejecuciones descartadas para estabilizar la GPU; luego 4 rondas ABBA, '
log += 'cada ejecucion con 1 calentamiento + 10 mediciones\n'
estabilizar = subprocess.run([PROGRAMA['global'], 'data/runway.pgm', '--runs', '2000'],
                             text=True, capture_output=True)
log += f'== estabilizacion (descartada): codigo {estabilizar.returncode}; '
log += estabilizar.stdout.splitlines()[-2] if estabilizar.returncode == 0 else estabilizar.stderr
log += '\n'
assert estabilizar.returncode == 0, estabilizar.stderr
tablas, estados = [], []
try:
    for ronda, variantes in ORDEN.items():
        for variante in variantes:
            estado = subprocess.check_output(CONSULTA, text=True).strip()
            estados.append(f'{ronda},{variante},{estado}')
            temporal = Path('build/fase4') / ronda / variante
            shutil.rmtree(temporal, ignore_errors=True)
            run = subprocess.run([PROGRAMA[variante], 'data/runway.pgm', '--runs', '10',
                                  '--output', str(temporal)], text=True, capture_output=True)
            salida = f'== {ronda} {variante} | GPU antes: {estado}\n{run.stdout}{run.stderr}'
            log += salida
            print(salida)
            assert run.returncode == 0, f'Fallo {variante} en {ronda}'
            destino = out / ronda / f'{variante}-times.csv'
            destino.parent.mkdir(exist_ok=True)
            shutil.copy(temporal / f'{variante}-times.csv', destino)
            tabla = pd.read_csv(destino)
            assert len(tabla) == 10 and tabla['validated'].all()
            assert tabla['run'].tolist() == list(range(1, 11))
            tablas.append(tabla.assign(ronda=ronda))
finally:
    (out / 'fase4-validacion.txt').write_text(log)
    (out / 'estado-gpu.csv').write_text('ronda,variant,pstate,sm_mhz,mem_mhz,temp_c,power_w\n'
                                       + '\n'.join(estados) + '\n')
tiempos = pd.concat(tablas, ignore_index=True)
display(tiempos.pivot_table(index='run', columns=['ronda', 'variant'], values='kernel_ms'))

## Comparación

Desviación estándar muestral, igual que en la fase 2. `diferencia_pct` es negativa si la versión constante es más rápida; `speedup` es tiempo global / tiempo constante. `p_welch` prueba si la diferencia de promedios es estadísticamente significativa. La fila `total` combina las 40 mediciones de cada versión.

In [ ]:
resumen = tiempos.groupby(['ronda', 'variant'])['kernel_ms'].agg(['count', 'mean', 'std', 'min', 'max'])
total = tiempos.groupby('variant')['kernel_ms'].agg(['count', 'mean', 'std', 'min', 'max'])
resumen = pd.concat([resumen, pd.concat({'total': total})])
resumen.to_csv(out / 'resumen.csv')
display(resumen)
filas = []
for ronda in [*ORDEN, 'total']:
    datos = tiempos if ronda == 'total' else tiempos.query('ronda == @ronda')
    g = datos.query('variant == "global"')['kernel_ms']
    c = datos.query('variant == "constante"')['kernel_ms']
    fila = dict(ronda=ronda, global_ms=g.mean(), constante_ms=c.mean(),
                diferencia_pct=100 * (c.mean() - g.mean()) / g.mean(), speedup=g.mean() / c.mean())
    try:
        from scipy.stats import ttest_ind
        fila['p_welch'] = ttest_ind(g, c, equal_var=False).pvalue
    except ImportError:
        pass
    filas.append(fila)
comparacion = pd.DataFrame(filas)
comparacion.to_csv(out / 'comparacion.csv', index=False)
display(comparacion)
display(pd.read_csv(out / 'estado-gpu.csv'))

## Resultados

In [ ]:
from google.colab import files
import zipfile
bundle = Path('build/fase4-constante.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.rglob('*')):
        if path.is_file():
            archive.write(path, path.as_posix())
files.download(str(bundle))